# ዘር · Zer — QLoRA fine-tuning on a free GPU

Fine-tune an open model on **Zer's full Amharic dataset** (~270k real
conversations + the app's knowledge bases) with 4-bit QLoRA.

**Works on:** Google Colab (free T4) and Kaggle (T4 / P100).

> **Use your free GPU within the provider's rules.** One account, one session
> at a time. Don't open parallel sessions or extra accounts to dodge limits.

### Pick a model by time budget (T4 16 GB)

The set is now ~270k examples; the free tiers won't do multiple epochs over all
of it, so cell 5 caps training at a balanced `SUBSET` (default 120k). Set
`SUBSET = 0` for the full set (needs a paid GPU / several sessions).

| Model | Fits | Rough time (1 epoch, ~120k) | Quality |
| --- | --- | --- | --- |
| `Qwen/Qwen2.5-1.5B-Instruct` | any T4 | ~3–6 h | good |
| `Qwen/Qwen2.5-3B-Instruct` | T4 | ~10–16 h (Kaggle 30 h/wk) | better |
| `Qwen/Qwen2.5-7B-Instruct` | T4, batch 1 | multi-session; needs resume | best |

Checkpoints are written to Google Drive (Colab) or `/kaggle/working` (Kaggle), so
you can **resume** after a session ends.

**After training:** download `zer-lora.zip` (or push to the HF Hub) and serve it
with vLLM / llama.cpp behind `LLM_BASE_URL` (see `training/README.md`).

In [ ]:
# 1) Confirm you actually have a GPU
!nvidia-smi || echo 'No GPU — Runtime ▸ Change runtime type ▸ T4 GPU'

In [ ]:
# 2) Get the code
%cd /content 2>/dev/null || %cd /kaggle/working
import os
if os.path.isdir('amharic-nlp-chatbot'):
    %cd amharic-nlp-chatbot && !git pull --ff-only || true
else:
    !git clone --depth 1 https://github.com/ZebraCodeX/amharic-nlp-chatbot.git
    %cd amharic-nlp-chatbot
print(os.getcwd())

In [ ]:
# 3) Install the training stack (torch is already present on Colab/Kaggle)
!pip install -q -U 'transformers<5' peft accelerate datasets bitsandbytes sentencepiece safetensors pyarrow
import torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('transformers', transformers.__version__, '| peft', peft.__version__,
      '| bnb', bitsandbytes.__version__)

### 4) Build the dataset (this is what the old notebook missed)

`build_dataset.py` alone only emits the ~500 seed rows. The fetch step below
pulls the free conversational/instruction corpora (AddisGPT, FineTome, translated
Alpaca + Dolly, EthioNLP tasks, Amharic GSM8K ≈ 270k) first, then
`build_dataset.py` merges them with Zer's knowledge base, rich answers,
dictionary, taught translations and codegen recipes.
This downloads ~1.2 GB, once.

In [ ]:
!python tools/fetch_conversation_corpus.py
!python tools/fetch_topics.py
!python training/build_dataset.py
!python training/clean_dataset.py
!wc -l training/data/amharic_sft.clean.jsonl
!head -c 300 training/data/amharic_sft.clean.jsonl

### 5) Split a held-out test set

We keep ~400 rows out of training so `eval_compare.py` can score the fine-tune
against the base model on data it never saw.

In [ ]:
import json, random
SUBSET = 120_000     # 0 = use every example; caps a free-GPU run to its budget
src = 'training/data/amharic_sft.clean.jsonl'
rows = [l for l in open(src, encoding='utf-8') if l.strip()]
random.Random(42).shuffle(rows)
N_TEST = 400
test, train = rows[:N_TEST], rows[N_TEST:]
if SUBSET:
    train = train[:SUBSET]
open('training/data/amharic_test.jsonl', 'w', encoding='utf-8').writelines(test)
open('training/data/amharic_train.jsonl', 'w', encoding='utf-8').writelines(train)
print('train', len(train), '| held-out test', len(test))

### 6) Configure and train

Set `MODEL` from the table above. Keep `BATCH=2, GA=8` for 1.5B/3B on a T4;
for 7B use `BATCH=1, GA=16`. `SAVE_STEPS` enables resumable checkpoints.

Set `RESUME=True` on a later run to continue from the last checkpoint in `OUT`.

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # bigger than the 1.5B shipped GGUF; see table
EPOCHS, BATCH, GA, MAXLEN, LR = 1, 2, 8, 1024, 2e-4
SAVE_STEPS = 500
RESUME = False
USE_DRIVE = True    # Colab only: persist checkpoints across sessions

import os
OUT = 'training/out/zer-lora'
if USE_DRIVE:
    try:
        from google.colab import drive  # noqa
        drive.mount('/content/drive')
        OUT = '/content/drive/MyDrive/zer-training/zer-lora'
        os.makedirs(OUT, exist_ok=True)
        DATASET = '/content/drive/MyDrive/zer-training/amharic_train.jsonl'
        import shutil
        if not os.path.exists(DATASET):
            shutil.copy('training/data/amharic_train.jsonl', DATASET)
    except Exception as e:
        print('Drive not used:', e)
        DATASET = 'training/data/amharic_train.jsonl'
else:
    DATASET = 'training/data/amharic_train.jsonl'

print('model =', MODEL)
print('out   =', OUT)

In [ ]:
RES = '--resume' if RESUME else ''
!HF_HUB_DISABLE_PROGRESS_BARS=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python training/train_qlora.py \
  --model {MODEL} --dataset {DATASET} --out {OUT} \
  --epochs {EPOCHS} --batch {BATCH} --grad-accum {GA} --max-len {MAXLEN} --lr {LR} \
  --lora-r 32 --lora-alpha 64 --save-steps {SAVE_STEPS} {RES}

In [ ]:
# 7) Merge the LoRA adapter into a standalone model
!python training/merge_adapter.py --base {MODEL} --adapter {OUT} \
  --out {OUT}-merged

In [ ]:
# 8) Quick sanity check: does it answer in the language of the prompt?
!python training/eval.py --model {OUT}-merged --limit 5

### 9) Score it — fine-tune vs the base model, on the held-out test set

`eval_compare.py` reports language-match, ROUGE-L, chrF and token-F1 against the
reference answers. You can add any locally-loadable model to `--models`
(e.g. `Qwen/Qwen2.5-0.5B-Instruct`) to compare. Closed frontier models have no
downloadable weights, so they can't be scored here — this is a same-hardware,
transparent comparison, not a world leaderboard.

In [ ]:
!python training/eval_compare.py \
  --models {OUT}-merged,{MODEL} \
  --test training/data/amharic_test.jsonl --limit 200 \
  --out eval_results.json

### 10) Keep the result — download the adapter and/or push to the Hugging Face Hub

The adapter is tiny (~30–150 MB). `build_dataset.py` outputs are reproducible
from the repo, so the adapter is all you need to carry to another machine.

In [ ]:
import os, shutil, glob
shutil.make_archive('zer-lora', 'zip', OUT)
print('adapter size (MB):', round(os.path.getsize('zer-lora.zip') / 1e6, 2))
print('merged files:', glob.glob(OUT + '-merged/*')[:4], '…')
try:
    from google.colab import files
    files.download('zer-lora.zip')
except Exception:
    print('Kaggle: outputs under /kaggle/working are saved automatically.')

In [ ]:
# Optional: publish the adapter so the laptop / server can pull it
# import os
# os.environ['HF_TOKEN'] = 'hf_...'   # token with WRITE access
# REPO = 'your-username/zer-qwen-lora'
# !python training/push_adapter.py --folder {OUT} --repo {REPO}

### 11) Serve it and connect the app

On a GPU host with the merged model or the adapter:

```bash
pip install vllm
vllm serve training/out/zer-lora-merged --served-model-name zer --port 8000
```

Then point the deployed app at it (no redeploy needed — see `training/serve.sh`
and `training/connect-fly.sh`):

```bash
flyctl secrets set LLM_BASE_URL=https://YOUR-GPU-HOST/v1 LLM_MODEL=zer LLM_API_KEY=...
```

Zer prefers the LLM for open/creative questions and keeps the offline Amharic
brain (code, knowledge, translation) as the fallback.